SESIÓN 3 Notebook 1 LA-03-Transform-data

Here is the link to the GITHUB Repo of the data:
- https://github.com/MicrosoftLearning/mslearn-databricks/tree/main/data

Here is the guide we're following:
https://microsoftlearning.github.io/mslearn-databricks/Instructions/Exercises/LA-03-Transform-data.html


In [0]:

import pandas as pd

#Create variables to implement schema during reading with pandas

column_names = [
    "SalesOrderNumber", "SalesOrderLineNumber", "OrderDate", 
    "CustomerName", "Email", "Item", "Quantity", "UnitPrice", "Tax"
]

dtype_dict = {
    "SalesOrderNumber": str,
    "SalesOrderLineNumber": int,
    "OrderDate": str, 
    "CustomerName": str,
    "Email": str,
    "Item": str,
    "Quantity": int,
    "UnitPrice": float,
    "Tax": float
}



In [0]:
#Reading every file individually, then concatenate them and convert into spark DF 

url = 'https://raw.githubusercontent.com/MicrosoftLearning/mslearn-databricks/main/data/2019.csv'

df_2019 = pd.read_csv(url , names=column_names, header=None, dtype=dtype_dict)

url = 'https://raw.githubusercontent.com/MicrosoftLearning/mslearn-databricks/main/data/2020.csv'

df_2020 = pd.read_csv(url, names=column_names, header=None, dtype=dtype_dict)

url = 'https://raw.githubusercontent.com/MicrosoftLearning/mslearn-databricks/main/data/2021.csv'

df_2021 = pd.read_csv(url, names=column_names, header=None, dtype=dtype_dict)

df_2019_2021 = pd.concat([df_2019,df_2020,df_2021],axis=0)

df_2019_2021["OrderDate"] = pd.to_datetime(df_2019_2021["OrderDate"], errors='coerce')

df = spark.createDataFrame(df_2019_2021)




In [0]:
 from pyspark.sql.functions import col
 df = df.dropDuplicates()
 df = df.withColumn('Tax', col('UnitPrice') * 0.08)
 df = df.withColumn('Tax', col('Tax').cast("float"))
 display(df.limit(100))

In [0]:
customers = df['CustomerName', 'Email']
print(customers.count())
print(customers.distinct().count())
display(customers.distinct())

In [0]:
customers = df.select("CustomerName", "Email").where(df['Item']=='Road-250 Red, 52')
print(customers.count())
print(customers.distinct().count())
display(customers.distinct())

In [0]:
productSales = df.select("Item", "Quantity").groupBy("Item").sum()
display(productSales)

In [0]:
from pyspark.sql.functions import year
yearlySales = df.select(year("OrderDate").alias("Year")).groupBy("Year").count().orderBy("Year")
display(yearlySales)

In [0]:
df.createOrReplaceTempView("salesorders")

In [0]:
%sql
    
SELECT YEAR(OrderDate) AS OrderYear,
       SUM((UnitPrice * Quantity) + Tax) AS GrossRevenue
FROM salesorders
GROUP BY YEAR(OrderDate)
ORDER BY OrderYear;